Chatgpt4-o

# Session 1: Word alignment
source_sentence, target_sentence 和 word_alignment 是定好的

In [ ]:
# 修改來源語句、目標語句和對齊語句範例 (中文到英文)
source_sentence = ["先生", "張三", "去了", "北京"]
target_sentence = ["Mr.", "Zhang San", "went", "to", "Beijing"]
word_alignment = {0: 0, 1: 1, 2: 3, 3: 4}  # 中文到英文的詞對齊結果

# 命名實體 (不應翻譯的專有名詞，例如姓名或地名)
named_entities = {"張三", "北京"}

def correct_translation(source_sentence, target_sentence, word_alignment, named_entities):
    corrected_sentence = target_sentence.copy()

    for source_index, target_index in word_alignment.items():
        # 確保目標索引在目標句的範圍內
        if target_index < len(target_sentence):
            source_word = source_sentence[source_index]
            if source_word in named_entities:
                # 如果目標單字是命名實體，則將其替換為來源單字
                corrected_sentence[target_index] = source_word

    return corrected_sentence

# 測試修正翻譯
corrected_sentence = correct_translation(source_sentence, target_sentence, word_alignment, named_entities)
print(corrected_sentence)


['Mr.', '張三', 'went', 'to', '北京']


在修正翻譯的工程之前，注意到 source_sentence 和 target_sentence 都是有被切割好的。

因此接下來對各個切割套件作觀察。

# Session 2: Slightly test of Term noun
測試以下句子：
* sentence1 = "張三先生去了北京"
* sentence2 = "我要去桃園"

## 環境安裝

In [ ]:
%%capture
# for visuzlization
!pip install deplacy

# for Trankit
!pip install trankit transformers

# for stanfordnlp
!pip install stanfordnlp

# for jieba
!pip install jieba

# for HanLP
!pip install hanlp-restful

# for Stanza
!pip install stanza

# for UDPipe2
def UDPipe2_nlp(t):
  import urllib.request,urllib.parse,json
  with urllib.request.urlopen("https://lindat.mff.cuni.cz/services/udpipe/api/process?model=zh_gsd&tokenizer&tagger&parser&data="+urllib.parse.quote(t)) as r:
    return json.loads(r.read())["result"]

# for esupar
!pip install esupar

# for NLP-Cube
!pip uninstall -y torchaudio torchvision
!pip install blinker --ignore-installed
!pip install nlpcube

# for spacy-udpipe
!pip install spacy-udpipe

# for UD-Chinese
!pip install udchinese

# for spacy
!pip install spacy-transformers
!python -m spacy download zh_core_web_trf

## Sentence 1

In [ ]:
sent1 = "張三先生去了北京"

### [Trankit](https://github.com/nlp-uoregon/trankit)

In [ ]:
import trankit
nlp=trankit.Pipeline("traditional-chinese")

Loading pretrained XLM-Roberta, this may take a while...
Loading tokenizer for traditional-chinese
Loading tagger for traditional-chinese
Loading lemmatizer for traditional-chinese
Loading NER tagger for traditional-chinese
Active language: traditional-chinese


In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 2.2 s, sys: 324 ms, total: 2.53 s
Wall time: 2.95 s


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc, port=None)

張   PROPN ═╗<╗   nmod
三   PROPN <╝ ║   flat:name
先生 NOUN  ═══╝<╗ nsubj
去   VERB  ═╗═╗═╝ root
了   PART  <╝ ║   case:aspect
北京 PROPN <══╝   obj


### [jieba](https://github.com/fxsjy/jieba)

In [ ]:
import jieba
tokenized_sentence = list(jieba.cut(sent1))

In [ ]:
print("Jieba 分詞結果:", tokenized_sentence)

Jieba 分詞結果: ['張三', '先生', '去', '了', '北京']


### [HanLP](https://github.com/hankcs/HanLP)

In [ ]:
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/api', auth=None, language='zh')
# 使用 HanLP 進行依存關係分析
result = HanLP.parse(sent1)

In [ ]:
tokenized_sentence = result['tok/fine']

print("HanLP 分詞結果:", tokenized_sentence)

HanLP 分詞結果: [['張三', '先生', '去', '了', '北京']]


### [Stanza](https://github.com/stanfordnlp/stanza)

In [ ]:
import stanza
nlp=stanza.Pipeline("zh-hant")

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json


INFO:stanza:Loading these models for language: zh-hant (Traditional_Chinese):
| Processor | Package      |
----------------------------
| tokenize  | gsd          |
| pos       | gsd_nocharlm |
| lemma     | gsd_nocharlm |
| depparse  | gsd_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 83.5 ms, sys: 2.58 ms, total: 86.1 ms
Wall time: 498 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   PROPN <══╗   nmod
三   NUM   <╗ ║   nummod
先生 NOUN  ═╝═╝<╗ nsubj
去   VERB  ═╗═╗═╝ root
了   AUX   <╝ ║   aux
北京 PROPN <══╝   obj


### [UDPipe 2](http://ufal.mff.cuni.cz/udpipe/2)


In [ ]:
%%time
doc=UDPipe2_nlp(sent1)

CPU times: user 16.5 ms, sys: 1.91 ms, total: 18.4 ms
Wall time: 817 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   PROPN <════╗   nmod
三   NUM   <╗   ║   nummod
先生 NOUN  ═╝═╗═╝<╗ nsubj
去   VERB  ═╗═════╝ root
了   AUX   <╝ ║     aux
北京 PROPN <══╝     nmod


### [esupar](https://github.com/KoichiYasuoka/esupar)


In [ ]:
import esupar
nlp=esupar.load("zh")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.50k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/407M [00:00<?, ?B/s]

supar.model:   0%|          | 0.00/462M [00:00<?, ?B/s]

Some weights of the model checkpoint at KoichiYasuoka/chinese-bert-wwm-ext-upos were not used when initializing BertModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at KoichiYasuoka/chinese-bert-wwm-ext-upos and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
%%time
doc=nlp(sent1)

INFO:supar:Loading the data
INFO:supar:
Dataset(n_sentences=1, n_batches=1, n_buckets=1)
INFO:supar:Making predictions on the dataset
INFO:supar:0:00:00.195880s elapsed, 5.11 Sents/s


CPU times: user 325 ms, sys: 2.56 ms, total: 327 ms
Wall time: 399 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   PROPN <══╗   nmod
三   PROPN <╗ ║   nmod
先生 NOUN  ═╝═╝<╗ nsubj
去   VERB  ═╗═╗═╝ root
了   AUX   <╝ ║   aux
北京 PROPN <══╝   obj


### [NLP-Cube](https://github.com/Adobe/NLP-Cube)


In [ ]:
from cube.api import Cube
nlp=Cube()
nlp.load("zh")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a m

In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 560 ms, sys: 6.25 ms, total: 567 ms
Wall time: 2.04 s


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   PROPN        root
三先 ADV   <════╗ nsubj
生去 VERB  ═╗═╗═╝ root
了   PART  <╝ ║   aux:aspect
北京 PROPN <══╝   obj


### [spacy-udpipe](https://github.com/TakeLab/spacy-udpipe)


In [ ]:
import spacy_udpipe
spacy_udpipe.download("zh")
nlp=spacy_udpipe.load("zh")

Downloaded pre-trained UDPipe model for 'zh' language


In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 6.94 ms, sys: 720 µs, total: 7.66 ms
Wall time: 19.2 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   PROPN ═══╗<╗ nsubj
三   NUM   <╗ ║ ║ nummod
先生 NOUN  ═╝<╝ ║ appos
去   VERB  ═╗═╗═╝ ROOT
了   PART  <╝ ║   case:aspect
北京 PROPN <══╝   obj


### [UD-Chinese](https://pypi.org/project/udchinese)


In [ ]:
import udchinese
nlp=udchinese.load()

In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 3.55 ms, sys: 0 ns, total: 3.55 ms
Wall time: 3.56 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張   VERB  <════════╗ nsubj
三   NUM   <╗       ║ nummod
先   NOUN  ═╝<════╗ ║ obl:lmod
生   VERB  ═╗═╗═╗═╝═╝ root
去   VERB  <╝ ║ ║     flat:vv
了   PART  <══╝ ║     case:aspect
北京 PROPN <════╝     obj


### [spaCy](https://spacy.io/)


In [ ]:
import spacy
nlp=spacy.load("zh_core_web_trf")

In [ ]:
%%time
doc=nlp(sent1)

CPU times: user 98.1 ms, sys: 435 µs, total: 98.6 ms
Wall time: 117 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

張三 PROPN <╗     compound:nn
先生 NOUN  ═╝<══╗ nsubj
去   VERB  ═╗═╗═╝ ROOT
了   PART  <╝ ║   aux:asp
北京 PROPN <══╝   dobj


### Results for sentence1


* "北京" 有切出來 ：Trankit, stanza, esupar, NLP-Cube, spacy_udpipe, udchinese

* "張三"、"北京" 都有切出來 ：Jieba, HanLP, UDPipe2_nlp, spacy
* 效果明顯比較不好：NLP-Cube [張/三先/生去]

## Sentence 2

In [ ]:
sent2 = "我要去桃園"

### [Trankit](https://github.com/nlp-uoregon/trankit)

In [ ]:
import trankit
nlp=trankit.Pipeline("traditional-chinese")

Loading pretrained XLM-Roberta, this may take a while...
Loading tokenizer for traditional-chinese
Loading tagger for traditional-chinese
Loading lemmatizer for traditional-chinese
Loading NER tagger for traditional-chinese
Active language: traditional-chinese


In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 1.85 s, sys: 176 ms, total: 2.03 s
Wall time: 2.08 s


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON  <════╗ nsubj
要   AUX   <══╗ ║ aux
去   VERB  ═╗═╝═╝ root
桃園 PROPN <╝     obj


### [jieba](https://github.com/fxsjy/jieba)

In [ ]:
import jieba
tokenized_sentence = list(jieba.cut(sent2))

In [ ]:
print("Jieba 分詞結果:", tokenized_sentence)

Jieba 分詞結果: ['我要', '去', '桃園']


### [HanLP](https://github.com/hankcs/HanLP)

In [ ]:
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/api', auth=None, language='zh')
# 使用 HanLP 進行依存關係分析
result = HanLP.parse(sent2)

In [ ]:
tokenized_sentence = result['tok/fine']

print("HanLP 分詞結果:", tokenized_sentence)

HanLP 分詞結果: [['我', '要', '去', '桃園']]


### [Stanza](https://github.com/stanfordnlp/stanza)

In [ ]:
import stanza
nlp=stanza.Pipeline("zh-hant")

INFO:stanza:Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES


INFO:stanza:Downloaded file to /root/stanza_resources/resources.json
INFO:stanza:Loading these models for language: zh-hant (Traditional_Chinese):
| Processor | Package      |
----------------------------
| tokenize  | gsd          |
| pos       | gsd_nocharlm |
| lemma     | gsd_nocharlm |
| depparse  | gsd_nocharlm |

INFO:stanza:Using device: cpu
INFO:stanza:Loading: tokenize
INFO:stanza:Loading: pos
INFO:stanza:Loading: lemma
INFO:stanza:Loading: depparse
INFO:stanza:Done loading processors!


In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 58.5 ms, sys: 33.2 ms, total: 91.8 ms
Wall time: 85.4 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON <════╗ nsubj
要   AUX  <══╗ ║ aux
去   VERB ═╗═╝═╝ root
桃園 NOUN <╝     obj


### [UDPipe 2](http://ufal.mff.cuni.cz/udpipe/2)


In [ ]:
%%time
doc=UDPipe2_nlp(sent2)

CPU times: user 21.9 ms, sys: 20.1 ms, total: 42 ms
Wall time: 804 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON <════╗ nsubj
要   AUX  <══╗ ║ aux
去   VERB ═╗═╝═╝ root
桃園 NOUN <╝     iobj


### [esupar](https://github.com/KoichiYasuoka/esupar)


In [ ]:
import esupar
nlp=esupar.load("zh")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at KoichiYasuoka/chinese-bert-wwm-ext-upos were not used when initializing BertModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertModel were not initialized from the model checkpoint at KoichiYasuoka/chinese-b

In [ ]:
%%time
doc=nlp(sent2)

INFO:supar:Loading the data
INFO:supar:
Dataset(n_sentences=1, n_batches=1, n_buckets=1)
INFO:supar:Making predictions on the dataset
INFO:supar:0:00:00.145988s elapsed, 6.85 Sents/s


CPU times: user 228 ms, sys: 7.21 ms, total: 235 ms
Wall time: 296 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON  <════╗ nsubj
要   AUX   <══╗ ║ aux
去   VERB  ═╗═╝═╝ root
桃園 PROPN <╝     obj


### [NLP-Cube](https://github.com/Adobe/NLP-Cube)


In [ ]:
from cube.api import Cube
nlp=Cube()
nlp.load("zh")

Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of the model checkpoint at xlm-roberta-base were not used when initializing XLMRobertaModel: ['lm_head.dense.weight', 'lm_head.bias', 'lm_head.dense.bias', 'lm_head.layer_norm.bias', 'lm_head.layer_norm.weight']
- This IS expected if you are initializing XLMRobertaModel from the checkpoint of a m

In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 441 ms, sys: 13.6 ms, total: 455 ms
Wall time: 686 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON      root
要   AUX  <══╗ aux
去桃 VERB <╗ ║ compound
園   PART ═╝═╝ root


### [spacy-udpipe](https://github.com/TakeLab/spacy-udpipe)


In [ ]:
import spacy_udpipe
spacy_udpipe.download("zh")
nlp=spacy_udpipe.load("zh")

Already downloaded a model for the 'zh' language


In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 5.13 ms, sys: 267 µs, total: 5.4 ms
Wall time: 21 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON  <════╗ nsubj
要   AUX   <══╗ ║ aux
去   VERB  ═╗═╝═╝ ROOT
桃園 PROPN <╝     obj


### [UD-Chinese](https://pypi.org/project/udchinese)


In [ ]:
import udchinese
nlp=udchinese.load()

In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 2.63 ms, sys: 0 ns, total: 2.63 ms
Wall time: 3.77 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我 PRON <══════╗ nsubj
要 VERB <════╗ ║ aux
去 VERB ═══╗═╝═╝ root
桃 NOUN <╗ ║     nmod
園 NOUN ═╝<╝     obj


### [spaCy](https://spacy.io/)


In [ ]:
import spacy
nlp=spacy.load("zh_core_web_trf")

In [ ]:
%%time
doc=nlp(sent2)

CPU times: user 165 ms, sys: 3.35 ms, total: 168 ms
Wall time: 310 ms


In [ ]:
import deplacy
deplacy.render(doc)
deplacy.serve(doc,port=None)

我   PRON  <════╗ nsubj
要   VERB  <══╗ ║ xcomp
去   VERB  ═╗═╝═╝ ROOT
桃園 PROPN <╝     dobj


### Results for sentence2


* 有把 "桃園" 切出來：trankit、Jieba、HanLP、stanza、UDPipe2_nlp、esupar、spacy_udpipe、spacy
* 沒有把 "桃園" 切出來：NLP-Cube、udchinese

# Session 3: 以 30 個繁體中文句子作測試

## 30 個繁體中文句子

In [ ]:
test_sentence = {
  "sentences": [
    "張三在台北101前面拍了很多照片。",
    "李四昨天在台中火車站迷路了。",
    "王五參加了鴻海公司的年度大會。",
    "華為手機在全球市場上的銷量持續增長。",
    "張三開了一間叫做「香格里拉」的餐廳。",
    "小明帶著他的iPhone去了華山藝文中心。",
    "你知道Apple的總部在哪裡嗎？",
    "昨天在星巴克遇見了老朋友林小明。",
    "上海是中國的經濟中心。",
    "Google正在開發一個新的AI模型。",
    "馬雲創辦的阿里巴巴是中國最大的電商平台之一。",
    "珠穆朗瑪峰是世界最高的山。",
    "高雄的六合夜市是美食愛好者的天堂。",
    "臺灣大學的學生來參加這次比賽。",
    "蘋果電腦的發明改變了整個科技產業。",
    "中國移動的用戶數量每年都在增加。",
    "香港的金融市場非常活躍。",
    "IBM推出了一款全新的量子計算機。",
    "Microsoft的Windows系統是全球最常用的操作系統之一。",
    "劉德華出演了很多經典的電影。",
    "台灣的玉山是當地最高的山峰。",
    "世界衛生組織在日內瓦總部召開會議。",
    "任天堂的Switch遊戲機在全球大受歡迎。",
    "中央電視台播放了關於新冠疫情的最新報導。",
    "奧林匹克運動會將在東京舉行。",
    "特斯拉的自動駕駛技術引發了廣泛關注。",
    "小紅書是一個非常受歡迎的社交平台。",
    "Facebook的隱私政策再次引發爭議。",
    "日本的富士山每年都吸引大量遊客。",
    "李小龍的武術精神影響了全世界。"
  ]
}


##Jieba

In [ ]:
import jieba

res = {}

for item in test_sentence["sentences"]:
  tokenized_sentence = list(jieba.cut(item))
  res[item] = list(tokenized_sentence)


Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 1.245 seconds.
DEBUG:jieba:Loading model cost 1.245 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


In [ ]:
import pandas as pd
df = pd.DataFrame(list(res.items()), columns=['原句', '切割後的句子'])

df.head()

,原句,切割後的句子
0,張三在台北101前面拍了很多照片。,"[張三在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]"
1,李四昨天在台中火車站迷路了。,"[李四, 昨天, 在, 台, 中, 火車, 站, 迷路, 了, 。]"
2,王五參加了鴻海公司的年度大會。,"[王五, 參加, 了, 鴻, 海, 公司, 的, 年度, 大會, 。]"
3,華為手機在全球市場上的銷量持續增長。,"[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]"
4,張三開了一間叫做「香格里拉」的餐廳。,"[張三開, 了, 一間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]"


In [ ]:
df.to_csv('jieba.csv')

## HanLP

In [ ]:
from hanlp_restful import HanLPClient
HanLP = HanLPClient('https://www.hanlp.com/api', auth=None, language='zh')

In [ ]:
import time
HanLP_res = {}
i=0
for item in (test_sentence["sentences"]):
  i+=1
  if i%2 ==0:
    result = HanLP.parse(item)
    tokenized_sentence = result['tok/fine']

    HanLP_res[item] = list(tokenized_sentence)
  else:
    time.sleep(60)

    result = HanLP.parse(item)
    tokenized_sentence = result['tok/fine']

    HanLP_res[item] = list(tokenized_sentence)

In [ ]:
import pandas as pd
HanLP_df = pd.DataFrame(list(HanLP_res.items()), columns=['原句', 'HanLP 分詞結果'])

HanLP_df.head()


,原句,HanLP 分詞結果
0,張三在台北101前面拍了很多照片。,"[[張三, 在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]]"
1,李四昨天在台中火車站迷路了。,"[[李四, 昨天, 在, 台中, 火車站, 迷路, 了, 。]]"
2,王五參加了鴻海公司的年度大會。,"[[王五, 參加, 了, 鴻海, 公司, 的, 年度, 大會, 。]]"
3,華為手機在全球市場上的銷量持續增長。,"[[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]]"
4,張三開了一間叫做「香格里拉」的餐廳。,"[[張三, 開, 了, 一, 間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]]"


In [ ]:
HanLP_df.to_csv('HanLP.csv')

## UDPipe2_nlp

In [ ]:
UD_res = {}

for item in test_sentence["sentences"]:
  result = UDPipe2_nlp(item)

  # 提取分詞結果
  tokens = []
  # 逐行處理
  for line in result.split('\n'):
    if line and line[0].isdigit():
      tokens.append(line.split('\t')[1])

  UD_res[item] = list(tokens)




In [ ]:
import pandas as pd
UD_df = pd.DataFrame(list(res.items()), columns=['原句', '切割後的句子'])

UD_df.head()

,原句,切割後的句子
0,張三在台北101前面拍了很多照片。,"[張三在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]"
1,李四昨天在台中火車站迷路了。,"[李四, 昨天, 在, 台, 中, 火車, 站, 迷路, 了, 。]"
2,王五參加了鴻海公司的年度大會。,"[王五, 參加, 了, 鴻, 海, 公司, 的, 年度, 大會, 。]"
3,華為手機在全球市場上的銷量持續增長。,"[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]"
4,張三開了一間叫做「香格里拉」的餐廳。,"[張三開, 了, 一間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]"


In [ ]:
UD_df.to_csv('UDPipe2_nlp.csv')

## spacy

In [ ]:
import spacy
nlp = spacy.load("zh_core_web_trf")

spacy_res = {}

for source_sentence in test_sentence["sentences"]:
  doc = nlp(source_sentence)
  tokenized_sentence = [token.text for token in doc]

  spacy_res[source_sentence] = list(tokenized_sentence)


In [ ]:
import pandas as pd
spacy_df = pd.DataFrame(list(spacy_res.items()), columns=['原句', '切割後的句子'])

spacy_df.head()

,原句,切割後的句子
0,張三在台北101前面拍了很多照片。,"[張三, 在, 台北, 101, 前面, 拍, 了, 很多, 照片, 。]"
1,李四昨天在台中火車站迷路了。,"[李四, 昨天, 在, 台中, 火車站, 迷路, 了, 。]"
2,王五參加了鴻海公司的年度大會。,"[王五參, 加, 了, 鴻海, 公司, 的, 年度, 大會, 。]"
3,華為手機在全球市場上的銷量持續增長。,"[華為, 手機, 在, 全球, 市場, 上, 的, 銷量, 持續, 增長, 。]"
4,張三開了一間叫做「香格里拉」的餐廳。,"[張, 三, 開, 了, 一, 間, 叫做, 「, 香格里拉, 」, 的, 餐廳, 。]"


In [ ]:
spacy_df.to_csv('spacy.csv')